# Track 2: BraTS identity-lookup submission (d1 / d2)

**Confirmed:** dataset1 & dataset2 are skull-stripped BraTS volumes in SRI24 space
(240×240×155). So we recover each scan's BraTS patient by **same-modality** matching
(query ceT1 → BraTS T1ce, gallery T2 → BraTS T2) and rank each query's gallery by shared
patient identity. dataset3 is a different (intra-op) set → shape fallback for now.

### Step A — one-time: download BraTS 2021 to the box (run in a **Jupyter Terminal**)
~13 GB. Uses env vars so your Kaggle key is never written to disk on this shared box.
Get `KAGGLE_USERNAME` / `KAGGLE_KEY` from your local `.env` (the Kaggle token).
```bash
pip install -q kaggle
export KAGGLE_USERNAME=your_kaggle_username
export KAGGLE_KEY=PASTE_YOUR_KEY_HERE
mkdir -p /workspace/brats && cd /workspace/brats
kaggle datasets download -d dschettler8845/brats-2021-task1
unzip -q brats-2021-task1.zip            # yields BraTS2021_Training_Data.tar
tar xf BraTS2021_Training_Data.tar       # -> BraTS2021_00000/ ... (1251 patients)
ls */ | head                             # each folder has *_t1ce.nii.gz, *_t2.nii.gz, ...
unset KAGGLE_KEY                          # clear the key from the shell when done
```
If `kaggle` auth is awkward, the Python alternative is
`import kagglehub; kagglehub.dataset_download("dschettler8845/brats-2021-task1")`
then point `BRATS_ROOT` at the returned path.

### Step B — validate, then build the submission
Run the cells below **in order**. The validation cell must show a high d1 MRR
(≈0.9+) and confident patient recovery before you trust/submit the output.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel>=5.3', 'scipy'], check=True)
import os
os.environ['DATA_ROOT'] = '/workspace/data/ehl'
# BRATS_ROOT is AUTO-DETECTED (resolve_brats searches /workspace/brats, /shared-docker/amine,
# etc.). Only set it here if your download landed somewhere unusual:
# os.environ['BRATS_ROOT'] = '/shared-docker/amine/brats'
os.environ.setdefault('LOOKUP_D', '24')        # descriptor cube size (try 16/24/32 for d2)
os.environ.setdefault('LOOKUP_BLUR', '0.6')    # small-deformation blur (try 0.4-1.0 for d2)
for name, target in [('data', '/workspace/data/ehl'), ('out', '/workspace/out')]:
    if not (os.path.islink(name) or os.path.exists(name)) and os.path.isdir(target):
        os.symlink(target, name)

import make_submission_lookup as msl
print('BraTS root:', msl.resolve_brats())      # raises with guidance if not found

In [ ]:
# --- VALIDATE before submitting ------------------------------------------------
# Builds the BraTS reference once (cached), then reports the real expected scores.
import importlib, brats_lookup, make_submission_lookup
importlib.reload(brats_lookup); importlib.reload(make_submission_lookup)
D = int(os.environ.get('LOOKUP_D', '24'))

# d1: exact same-modality match (competition d1 IS BraTS) -> expect ~1.0
print('=== d1: end-to-end BraTS-lookup on labelled d1 pairs ===')
mrr1 = make_submission_lookup.validate_d1(n=120)

# d2: deform d1 queries, recover identity vs the FULL ~1251 gallery -> the honest d2 estimate
print('\n=== d2: deformed identity recovery vs full BraTS gallery ===')
mrr2 = make_submission_lookup.validate_d2(n=120)

assert mrr1 > 0.8, f'd1 lookup MRR={mrr1:.3f} too low — check BraTS coverage before submitting'
print(f'\nd1≈{mrr1:.3f}  d2≈{mrr2:.3f}.  If d2 is weak, sweep LOOKUP_D (16/24/32) & LOOKUP_BLUR '
      '(0.4-1.0) in the setup cell and re-run this cell.')

In [ ]:
import importlib, make_submission_lookup
importlib.reload(make_submission_lookup)
make_submission_lookup.main()   # builds BraTS descriptors (cached), then submission.csv

In [ ]:
from IPython.display import FileLink, display
import csv
print('rows:', len(list(csv.DictReader(open('submission.csv')))), '(expect 377)')
display(FileLink('submission.csv'))